# பாடம 12 - ஏஜென்ட் ஸ்கிராட்ச்பேட் மூலம் உரையாடல் வரலாறு சுருக்கம்

Microsoft Agent Framework பயன்படுத்தி நீண்ட உரையாடல்களில் சூழலை எப்படி நிர்வகிக்க வேண்டும் என்பதைக் காட்டு இந்த நோட்ட்புக் உதவுகிறது. உரையாடல்கள் அதிகமானபோது, டோக்கன் எண்ணிக்கை বৃদ্ধি பெறுகிறது — இறுதியில் மாதிரியின் சூழல் ஜன்னலை மீறும். இதை நாம் **சூழல் சுருக்க முறைமை** மற்றும் **ஏஜென்ட் ஸ்கிராட்ச்பேட்** எனும் தொடர்ச்சியான நினைவகத்துடன் கையாள்கிறோம்.

## நாம் கற்கப்போகும் விஷயங்கள்:
1. **சூழல் நிர்வகிப்பின் முக்கியத்துவம்**: டோக்கன் வரம்புகள் மற்றும் சூழல் ஜன்னல்களை புரிந்து கொள்வது
2. **சூழல் அறிவுள்ள ஏஜென்ட்கள்**: தங்களுடைய உரையாடல் சூழலை நிர்வகிக்கும் ஏஜென்ட்களை உருவாக்குதல்
3. **சூழல் சுருக்க முறைமை**: உரையாடல் வரலாரை சுருக்குவதற்கான கருவிகளைப் பயன்படுத்துதல்
4. **ஏஜென்ட் ஸ்கிராட்ச்பேட்**: சூழல் சுருக்கத்தின் போது தொடரும் நினைவகம்

## முன்னேற்பாடுகள்:
- Azure OpenAI அமைப்பு சுற்று மாறிலிகள் உள்ளமைவு
- முந்தைய பாடங்களில் அடிப்படை ஏஜென்ட் கொள்வனவு புரிதல்


## அமைப்பு


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## ஏன் சூழல் மேலாண்மை முக்கியம்

ஒவ்வொரு LLM க்கும் ஒரு முடிவுள்ள **சூழல் விண்டோ** உள்ளது — அது ஒரே கோரிக்கையில் செயலாக்கக்கூடிய அதிகபட்ச டோக்கன் எண்ணிக்கை. ஒரு பல-முறை உரையாடல் முன்னேறும் போது:

- **டோக்கன் எண்ணிக்கை ஒழுங்காக அதிகரிக்கிறது** ஒவ்வொரு பயனர் செய்தியுடன் மற்றும் உதவியாளர் பதிலுடன்.
- **சூழல் டோக்கன்கள் செலவின் முதன்மை காரணமாக இருப்பது** ஏனெனில் முழு வரலாறு ஒவ்வொரு முறையும் மீண்டும் அனுப்பப்படுகிறது.
- இறுதியில் உரையாடல் **சூழல் விண்டோவை மீறும்** மற்றும் மாடல் அல்லது குறைக்கிறது அல்லது பிழை காட்டுகின்றது.

### சூழல் மேலாண்மைக்கான நடைமுறைகள்

| நடைமுறை | அது எப்படி செயல்படுகிறது | உடன்பாடு |
|---|---|---|
| **குறைத்தல்** | பழைய செய்திகளை எறிதல் | ஆரம்ப சூழலை இழந்துவிடுகின்றது |
| **சுருக்கல்** | பழைய செய்திகளை சுருக்கம் செய்யுதல் | சில விவரங்கள் இழக்கப்படுகின்றன, ஆனால் முக்கிய அம்சங்கள் பிடிக்கப்பட்டுள்ளன |
| **ஸ்கிராட்ச்பேட் / வெளிப்புற நினைவகம்** | முக்கிய உண்மைகள் உரையாடலின் வெளியில் சேமிக்கப்படும் | கருவி அழைப்புகள் தேவை, ஆனால் எந்தக் குறைவையும் தாங்கிக் கொள்கிறது |

இந்த நோட்புக்கில் நாம் **சுருக்கலுடன்** **ஸ்கிராட்ச்பேட் கருவியை** ஒருங்கிணைக்கிறோம், ஆகையால் முகவர் உரையாடல் வரலாறு சுருக்கப்பட்டாலும் தொடர்ச்சியை காக்க முடியும்.


## контекстம் அறிந்த முகவரியை உருவாக்குதல்


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## நீண்ட உரையாடலை சிமுலேட் செய்வது

பரஸ்பர உரையாடல் மூலம் சூழலியல் எப்படி சேர்க்கப்படும் என்பதை பார்க்கலாம். முகவர் முக்கிய விவரங்களை (விருப்பங்கள், பட்ஜெட், பயணத் தேதிகள்) உரையாடல் சுழலில் வைத்திருத்து தொடர்ச்சியை ஒளிப்படுத்த வேண்டும்.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

எஜன் முந்தைய சுற்றுகளில் உள்ள சூழலை எப்படி வைத்திருப்பதை கவனியுங்கள் — அது ஜப்பான், சுஷி, கோயில்கள், புகைப்படம், $3000 பட்ஜெட், தனிமண் பயணம் மற்றும் ஏப்ரல் காலத்தைப் பற்றி அறிவது. குறுகிய உரையாடலில் இது நன்றாக வேலை செய்கிறது, ஆனால் உரையாடல் வளரும்போது முழு வரலாறு மீண்டும் அனுப்ப விலையுயர்ந்ததாக மாறுகிறது.

சூழலைப் பெருக்குவதற்காக அதிக சுற்றுகளை கொண்டு உரையாடலை தொடருவோம்:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

உரையாடல் வளர விரும்பும் போது, நாங்கள் சார்ந்துள்ள **சுருக்கல் கருவி**-ஐப் பயன்படுத்தி சார்ந்துள்ள சூழலை சுருக்கப்பட்ட வடிவில் ஒருங்கிணைக்க முடியும். முகவர் இந்த கருவியை முக்கிய விருப்பங்களை பதிவு செய்ய அழைக்கிறார், அதனால் பழைய செய்திகள் நீக்கப்பட்டாலும் முக்கியமான தகவல் பாதுகாக்கப்படுகிறது.

இந்த மாதிரி மேம்படுத்தப்பட்ட வரலாற்று குறைப்பு செய்வதற்கான அடித்தள கட்டு ஆகும்:
1. முகவர் உரையாடலிலிருந்து முக்கிய உண்மைகளை கண்டறிகிறார்
2. அவற்றை பதிலாகப் பராமரிக்க சுருக்கல் கருவியை அழைக்கிறார்
3. பழைய செய்திகள் பாதுகாப்பாக நீக்கப்படும், ஏனெனில் சுருக்கம் முக்கியத்துவம் வாய்ந்தவற்றை பிடிக்கிறது

கீழே, முகவர் கற்றுக்கொண்டதை சுருக்கமாக பதிவு செய்யச் செல்லக்கூடிய `summarize_preferences` கருவியை வரையறுக்கிறோம்.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework ஐ பயன்படுத்தி நீண்ட நேரம் நடக்கும் முகவர் உரையாடல்களில் சூழலை எப்படி நிர்வகிப்பது என்பதை கற்றுக்கொண்டீர்கள்:

### முக்கியக் கருத்துக்கள்
- **சூழல் ஜன்னல்கள் எல்லையிடம்** — உரையாடல் வரலாற்றிலுள்ள ஒவ்வொரு டோக்கனுக்கும் செலவு உள்ளது மற்றும் அது வரம்புக்குள் பொருந்தும்.
- **சுருக்கல் கருவிகள்** முகவருக்கு சேர்க்கப்பட்ட சூழலை சுருக்கப்பட்ட சுருக்கங்களில் சுருக்க உதவுகின்றன, அதில் திறன் பயன்படுத்தல் குறையும் ஆனால் முக்கியத் தகவல்கள் பாதுகாக்கப்படும்.
- **முகவர் ஸ்கிராட்ச்பாட்** நிலையான வெளிப்புற நினைவகம் வழங்குகிறது, இது எந்த உரையாடல் குறைப்பினாலும் நிலைத்திருக்கிறது.

### நீங்கள் உருவாக்கியது
- பல-துணை உரையாடல்களில் தொடர்ச்சியைக் காக்கும் **சூழல் அறிவுள்ள முகவர்**
- முக்கிய பயனர் விவரங்களை சுருக்கப்பட்ட வடிவில் பதிவு செய்யும் **சுருக்கல் கருவி** (`summarize_preferences`)
- சூழல் பாதுகாப்பும் மாற்றங்களை கையாள்வதும் காட்டும் **பல-துணை உரையாடல்**

### உலகின் நடைமுறை பயன்பாடுகள்
- **வாடிக்கையாளர் சேவை பாட்டுகள்**: நீண்ட ஆதரவு அமர்வுகளில் விருப்பங்களை நினைவில் வைக்க
- **தனிப்பட்ட உதவியாளர்கள்**: மீண்டும் சூழலை விளக்காமல் தொடர்ந்து நடைபெறும் திட்டங்களை கொணர்ந்து நோக்க
- **கல்வி ஆசான்கள்**: பல தொடர்புகளில் மாணவர்களின் முன்னேற்றத்தை பராமரி

### அடுத்த படிகள்
- கோப்பு அடிப்படையிலான நிலைத்தன்மையுடன் முழு ஸ்கிராட்ச்பாட் கருவியை நடைமுறைப்படுத்துதல்
- சுருக்கத்துக்குப் பின் தானாக வரலாறு குறைப்பு சேர்த்தல்
- அர்த்தமான நினைவகத் தேடலுக்கு வெக்டர் தரவுத்தளங்களுடன் இணைத்தல்
- முழு சூழலுடன் பல நாட்களுக்கு பிறகு உரையாடல்களை தொடரக்கூடிய முகவர்களை உருவாக்குதல்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**விரிவுரை**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சிக்கிறோம் என்றாலும், தானியங்கி மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் থাকতে வாய்ப்பு உள்ளது. அசல் ஆவணம் அதன் இயல்பான மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் தவறான புரிதல்கள் அல்லது தவறான அர்த்தங்கள் குறித்து நாங்கள் பொறுப்பேற்கவில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
